# Creating the vector store

* Create a postgress instance with the following characteristics:

In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.vectorstores import InMemoryVectorStore

import models.embedding_models.ollama_models

from langchain_postgres import PGVectorStore
from langchain_postgres import PGEngine

load_dotenv()
load_dotenv(".env.local", override=True)

connection_string = (
    f"postgresql+psycopg://{os.environ['POSTGRES_APP_USER']}:"
    f"{os.environ['POSTGRES_APP_PASSWORD']}@"
    f"{os.getenv('POSTGRES_HOST', 'localhost')}:"
    f"{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'ai_tutorial_rag')}"
)
engine = PGEngine.from_connection_string(connection_string)

engine.init_vectorstore_table(
    table_name="documents",
    vector_size=1024,
)

vector_store = PGVectorStore.create_sync(
    embedding_service=models.embedding_models.ollama_models.qwen3_embedding_model,
    engine=engine,
    table_name="documents"
)


# Storing then embedding documents

In [5]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Java supports virtual threads.",
        metadata={"topic": "java"}
    ),
    Document(
        page_content="Python supports list comprehensions.",
        metadata={"topic": "python"}
    ),
    Document(
        page_content="LangGraph is used for building stateful agent workflows.",
        metadata={"topic": "langgraph"}
    ),
    Document(
        page_content="LangChain provides abstractions for working with LLMs.",
        metadata={"topic": "langchain"}
    )
]
vector_store.add_documents(documents)

['cb13c76b-485f-4a80-9a53-16e45cee8b1d',
 '25d12a37-35b3-489a-a672-d1690e295044',
 '291cd83f-3e2d-4dad-9dfa-56d7bf34011e',
 '41babbbf-a936-40bd-8607-c54a083017a2']

# Retrieve the documents using similarity search

In [6]:
question = "What is LangGraph?"
results = vector_store.similarity_search(
    question,k=2
)

for doc in results:
    print(doc)

page_content='LangGraph is used for building stateful agent workflows.' metadata={'topic': 'langgraph'}
page_content='LangChain provides abstractions for working with LLMs.' metadata={'topic': 'langchain'}


# Retrieve the documents using retriever

In [7]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


docs = retriever.invoke(question)
for doc in results:
    print(doc)

page_content='LangGraph is used for building stateful agent workflows.' metadata={'topic': 'langgraph'}
page_content='LangChain provides abstractions for working with LLMs.' metadata={'topic': 'langchain'}
